In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("marquis03/cats-and-dogs")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'cats-and-dogs' dataset.
Path to dataset files: /kaggle/input/cats-and-dogs


In [30]:
import shutil
import os

source_dir = path
destination_dir = "/content/Dataset1"

if os.path.exists(destination_dir):
    print(f"{destination_dir} already exists. Removing it to start fresh...")
    shutil.rmtree(destination_dir)

shutil.copytree(source_dir, destination_dir)
print(f"Dataset successfully copied from {source_dir} to {destination_dir}")


Dataset successfully copied from /kaggle/input/cats-and-dogs to /content/Dataset1


In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense


In [15]:
import numpy as np
import matplotlib.pyplot as plt
import os
import cv2

#Data Augmentation

In [31]:
train_path = os.path.join("/content/Dataset1", "train")
test_path = os.path.join("/content/Dataset1", "val")


In [32]:
category = ['cat','dog']
category_info = {}
for cate in category:
  cate_path = os.path.join(train_path,cate)
  list_images = os.listdir(cate_path)
  print(f"{cate}:{len(list_images)}")
  category_info[cate] = [len(list_images),cate_path]

cat:95
dog:180


In [33]:
def DataAugmentation(cate_info):
    #Counting max image count
    max_count = max(c[0] for c in cate_info.values())
    #Augmentation image generator
    datagen = ImageDataGenerator(
        rotation_range=30,
        zoom_range=0.2,
        horizontal_flip=True,
        width_shift_range=0.2,
        height_shift_range=0.2,
        brightness_range=[0.7, 1.3],
        fill_mode="nearest")
    #loop for max category
    for cate in category:
      count, cate_path = cate_info[cate]
      required = max_count - count
      if required > 0:
        images = []
        #loop for reading all images
        for file in os.listdir(cate_path):
          img = cv2.imread(os.path.join(cate_path, file))
          if img is not None:
            img = cv2.resize(img, (224,224))
            images.append(img)
        #Generator for image generation
        images = np.array(images)
        generator = datagen.flow(
            images,
            batch_size = 1,
            save_to_dir = cate_path,
            save_prefix = "aug",
            save_format = 'jpg'
        )
        #loop for generating required images
        for i in range(required):
          next(generator)
          print(f"Image {i+1} Generated for category:{cate}")
DataAugmentation(cate_info=category_info)

Image 1 Generated for category:cat
Image 2 Generated for category:cat
Image 3 Generated for category:cat
Image 4 Generated for category:cat
Image 5 Generated for category:cat
Image 6 Generated for category:cat
Image 7 Generated for category:cat
Image 8 Generated for category:cat
Image 9 Generated for category:cat
Image 10 Generated for category:cat
Image 11 Generated for category:cat
Image 12 Generated for category:cat
Image 13 Generated for category:cat
Image 14 Generated for category:cat
Image 15 Generated for category:cat
Image 16 Generated for category:cat
Image 17 Generated for category:cat
Image 18 Generated for category:cat
Image 19 Generated for category:cat
Image 20 Generated for category:cat
Image 21 Generated for category:cat
Image 22 Generated for category:cat
Image 23 Generated for category:cat
Image 24 Generated for category:cat
Image 25 Generated for category:cat
Image 26 Generated for category:cat
Image 27 Generated for category:cat
Image 28 Generated for category:cat
I

In [34]:
train_gen = ImageDataGenerator(rescale=1./255)
val_gen = ImageDataGenerator(rescale=1./255)

In [35]:
train_data = train_gen.flow_from_directory(
    train_path,
    target_size=(224,224),
    batch_size=16,
    class_mode='binary'
)

Found 360 images belonging to 2 classes.


In [36]:
val_data = val_gen.flow_from_directory(
    test_path,
    target_size=(224,224),
    batch_size=16,
    class_mode='binary'
)

Found 70 images belonging to 2 classes.


In [37]:
train_data.class_indices

{'cat': 0, 'dog': 1}

In [38]:
cnn_model = Sequential()
cnn_model.add(Conv2D(64, (3,3), activation='relu', input_shape=(224,224,3)))
cnn_model.add(MaxPooling2D((2,2)))
cnn_model.add(Conv2D(64, (3,3), activation='relu'))
cnn_model.add(MaxPooling2D((2,2)))
cnn_model.add(Conv2D(32, (3,3), activation='relu'))
cnn_model.add(MaxPooling2D((2,2)))
cnn_model.add(Conv2D(16, (3,3), activation='relu'))
cnn_model.add(MaxPooling2D((2,2)))


/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [39]:
cnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 32)     │        18,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 16)     │         4,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 12, 16)     │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,808 (241.44 KB)

 Trainable params: 61,808 (241.44 KB)

 Non-trainable params: 0 (0.00 B)

In [40]:
cnn_model.add(Flatten())
cnn_model.add(Dense(128, activation='relu'))
cnn_model.add(Dense(64, activation='relu'))
cnn_model.add(Dense(32, activation='relu'))
cnn_model.add(Dense(16, activation='relu'))
cnn_model.add(Dense(1, activation='sigmoid'))

In [41]:
cnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 32)     │        18,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 16)     │         4,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 12, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2304)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       295,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 367,729 (1.40 MB)

 Trainable params: 367,729 (1.40 MB)

 Non-trainable params: 0 (0.00 B)

In [42]:
cnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy'])

In [43]:
cnn_model.fit(
    train_data,
    epochs=50,
    batch_size =16,
    verbose = 1,
    validation_data=val_data,
)

Epoch 1/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 30s 755ms/step - accuracy: 0.5083 - loss: 0.7068 - val_accuracy: 0.6571 - val_loss: 0.6877
Epoch 2/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 55ms/step - accuracy: 0.5000 - loss: 0.6935 - val_accuracy: 0.6571 - val_loss: 0.6881
Epoch 3/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step - accuracy: 0.5111 - loss: 0.6867 - val_accuracy: 0.6429 - val_loss: 0.6748
Epoch 4/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 55ms/step - accuracy: 0.6167 - loss: 0.6663 - val_accuracy: 0.6429 - val_loss: 0.6505
Epoch 5/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 58ms/step - accuracy: 0.6333 - loss: 0.6448 - val_accuracy: 0.6571 - val_loss: 0.6675
Epoch 6/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 69ms/step - accuracy: 0.7278 - loss: 0.5328 - val_accuracy: 0.6143 - val_loss: 0.6781
Epoch 7/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 73ms/step - accuracy: 0.6917 - loss: 0.5484 - val_accuracy: 0.6571 - val_loss: 0.6823
Epoch 8/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 54ms/step - accuracy: 0.6972 - loss: 0.5460 - val_accuracy: 0.6571 -

In [ ]:
import cv2

In [44]:
def predict_image(img):
  image = cv2.imread(img)
  image = cv2.resize(image, (224,224))
  image = np.expand_dims(image, axis=0)
  image = image / 255.0
  pred = cnn_model.predict(image)
  return pred

In [47]:
pred_val = predict_image("/content/dog1.jpg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step


In [48]:
if pred_val >= 0.5:
  print("dog")
else:
  print("cat")

dog
